# 07 — Báo cáo tổng hợp

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/07_final_report.ipynb)

**Không cần GPU.** Runtime → CPU. Khoảng 1 phút.

## Mục đích

Gộp mọi thứ đã chạy — 5 nấc của chuỗi (notebook 01–05) và các ô của ma trận tổ hợp
(notebook 06) — thành một báo cáo duy nhất:

1. **Bảng thang bậc** — mỗi nấc và phần tăng thêm so với nấc ngay trước.
2. **Kiểm định McNemar** cho từng bước leo thang.
3. **Phân tích theo độ phức tạp và theo nguồn dữ liệu** — kỹ thuật nào giúp loại bài nào.
4. **Chuyển dịch kiểu lỗi** qua các nấc.
5. **Đối chiếu mốc tham chiếu** — `PA_loose` so với con số tham chiếu.
6. Biểu đồ + hai bảng CSV để dán thẳng vào bài.

Notebook này **không chạy model**, chỉ đọc lại kết quả đã lưu, nên chạy lại bao nhiêu lần
cũng được và nấc nào chưa chạy thì tự bỏ qua.

## §1. Môi trường

In [ ]:
%%capture
!pip install -q pandas matplotlib

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
ON_KAGGLE = os.path.isdir("/kaggle/input")
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
elif ON_KAGGLE:
    for _r, _d, _f in os.walk("/kaggle/input"):
        if "vinumqa" in _d and "data" in _d:
            REPO_DIR = _r; _pinned = True; break
    OUTPUT_DIR = "/kaggle/working/vinumqa_runs"
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, stats

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

LADDER = [
    ("01_plain",           "Nấc 1 — inference thông thường"),
    ("02_prompt_eng",      "Nấc 2 — + prompt engineering"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("04_selfeval_sft",    "Nấc 4b — + self-eval (model SFT)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05_ace_sft",         "Nấc 5b — + ACE (model SFT)"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("06_comb_F_A",        "Tổ hợp — SFT + ACE (không self-eval)"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, csv, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "csv":   f"{stage}_program.csv",
        "meta":  f"{stage}_meta.json"}[kind])


def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: 3 file chuẩn + 1 file output thô."""
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_details_csv(rows, stage_path(stage, "csv"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'csv')}   ← 6 cột, mở bằng Excel được")
        print(f"      {stage_path(stage, 'meta')}")


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else ("Kaggle" if ON_KAGGLE else "local")
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + csv + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

## §2. Nạp kết quả các nấc

Notebook nào chưa chạy thì bỏ qua, phần còn lại vẫn báo cáo được.

In [ ]:
R, M, META = {}, {}, {}
for stage, label in LADDER:
    rows = load_stage(stage, quiet=True)
    if rows is None:
        print(f"  ⊘ {stage:<22} chưa có kết quả")
        continue
    R[stage] = rows
    M[stage] = pipeline.summarize(rows, label)
    _mp = stage_path(stage, "meta")
    META[stage] = json.load(open(_mp, encoding="utf-8")) if os.path.exists(_mp) else {}
    print(f"  ✓ {stage:<22} {len(rows)} mẫu | EA={M[stage]['EA']:.4f}")

assert R, "Chưa có nấc nào được chạy."

## §3. Bảng thang bậc

In [ ]:
_main = [s for s, _ in LADDER if s in R and not s.startswith("05_ace_random")]

print(f"\n{'═'*104}\n  KẾT QUẢ THEO NẤC — Qwen3-8B, ViNumQA test ({len(test_all)} mẫu)\n{'═'*104}")
print(f"{'nấc':<34}{'EA':>9}{'PA_strict':>11}{'PA_loose':>11}{'no_prog':>10}"
      f"{'exec_fail':>11}{'phút':>8}")
print("-" * 104)
for s in _main:
    m, meta = M[s], META.get(s, {})
    print(f"{m['label']:<34}{m['EA']:>9.4f}{m['PA_strict']:>11.4f}{m['PA_loose']:>11.4f}"
          f"{m['no_program']:>10.4f}{m['exec_none']:>11.4f}"
          f"{meta.get('metrics', {}).get('minutes', 0):>8.1f}")

print(f"\n{'─'*104}\n  PHẦN TĂNG THÊM CỦA TỪNG KỸ THUẬT\n{'─'*104}")
print(f"{'bước leo thang':<46}{'ΔEA':>10}{'ΔPA_strict':>13}{'cộng dồn ΔEA':>15}")
_base = M[_main[0]]["EA"] if _main else 0
for a, b in zip(_main, _main[1:]):
    if a.startswith("04_selfeval") and b.startswith("05_ace") and a[-4:] != b[-4:]:
        continue                     # không so chéo base ↔ sft
    d_ea = M[b]["EA"] - M[a]["EA"]
    d_pa = M[b]["PA_strict"] - M[a]["PA_strict"]
    print(f"{M[a]['label'][:20]} → {M[b]['label'][:22]:<24}"
          f"{d_ea:>10.4f}{d_pa:>13.4f}{M[b]['EA']-_base:>15.4f}")

## §4. Kiểm định từng bước leo thang

Các nấc chạy trên **cùng 497 mẫu** nên đây là dữ liệu *cặp*, và kiểm định đúng là
**McNemar**: chỉ nhìn các mẫu hai nấc bất đồng, đếm `b` (chỉ nấc trước đúng) và `c` (chỉ nấc
sau đúng). Kèm khoảng tin cậy 95 % bootstrap lấy mẫu lại theo cặp.

In [ ]:
print(f"\n{'═'*90}\n  KIỂM ĐỊNH (McNemar theo cặp, n={len(test_all)})\n{'═'*90}")
comparisons = []
_pairs = [(a, b) for a, b in zip(_main, _main[1:])
          if not (a.startswith("04_selfeval") and b.startswith("05_ace")
                  and a[-4:] != b[-4:])]
if "05_ace_base" in R and "05_ace_random_base" in R:
    _pairs.append(("05_ace_random_base", "05_ace_base"))

for a, b in _pairs:
    for key in ("ea", "pa_strict"):
        comparisons.append(stats.compare_pair(
            R[a], R[b], key=key,
            label=f"{M[b]['label']}  so với  {M[a]['label']}",
            name_base=a, name_variant=b))

_sig = [c for c in comparisons if c["p_value"] < 0.05 and c["key"] == "ea"]
print(f"\n  {len(_sig)}/{sum(1 for c in comparisons if c['key']=='ea')} bước leo thang "
      f"đạt p < 0.05 trên EA.")
print(f"  Với n={len(test_all)}, chênh lệch dưới ~2 điểm EA thường chưa đạt mức đó —")
print(f"  đó là giới hạn cỡ mẫu, không phải kỹ thuật thất bại.")

## §5. Kỹ thuật nào giúp loại bài nào

In [ ]:
_by_id = {s["id"]: s for s in test_all}

print(f"\n{'═'*92}\n  EA THEO ĐỘ PHỨC TẠP (số phép toán của gold)\n{'═'*92}")
_ks = sorted({r["n_ops_gold"] for r in next(iter(R.values())) if r["n_ops_gold"] > 0})
print(f"{'nấc':<34}" + "".join(f"{str(k) + ' phép':>10}" for k in _ks))
for s in _main:
    line = f"{M[s]['label']:<34}"
    for k in _ks:
        sub = [r for r in R[s] if r["n_ops_gold"] == k]
        line += f"{sum(r['ea'] for r in sub)/len(sub):>10.1%}" if sub else f"{'—':>10}"
    print(line)

print(f"\n{'═'*92}\n  EA THEO NGUỒN DỮ LIỆU\n{'═'*92}")
print(f"{'nấc':<34}{'FinQA-Vi':>12}{'Vi Data':>12}{'chênh':>10}")
for s in _main:
    g = {}
    for src in ("FinQA-Vi", "ViData"):
        sub = [r for r in R[s] if data.source_of(_by_id[r["id"]]) == src]
        g[src] = sum(r["ea"] for r in sub) / len(sub) if sub else 0
    print(f"{M[s]['label']:<34}{g['FinQA-Vi']:>12.1%}{g['ViData']:>12.1%}"
          f"{g['ViData']-g['FinQA-Vi']:>+10.1%}")

print(f"\n  Vi Data dùng table_* dày hơn hẳn FinQA-Vi; chênh lệch giữa hai cột cho biết")
print(f"  kỹ thuật nào giúp được nhóm câu đọc bảng.")

## §6. Chuyển dịch kiểu lỗi qua các nấc

In [ ]:
_outs = sorted({o for s in _main for o in M[s]["outcome"]})
print(f"\n{'═'*100}\n  PHÂN BỐ KẾT CỤC\n{'═'*100}")
print(f"{'nấc':<34}" + "".join(f"{o[:14]:>16}" for o in _outs))
for s in _main:
    print(f"{M[s]['label']:<34}" +
          "".join(f"{M[s]['outcome'].get(o, 0):>16}" for o in _outs))
print(f"\n  Hai cột đáng theo dõi nhất: 'khong_co_program' (model không sinh nổi khối")
print(f"  plaintext) và 'program_khong_chay_duoc' (sinh được nhưng executor từ chối).")
print(f"  Cả hai giảm dần là dấu hiệu các ràng buộc định dạng đang có tác dụng.")

## §7. Đối chiếu mốc tham chiếu

In [ ]:
_ref = io_utils.BASELINE_RESULTS["Qwen3-8B"]
_pub_csv = os.path.join(REPO_DIR, "reference", "baseline_results", "Qwen3-8B_program.csv")

print(f"{'═'*88}\n  SO VỚI MỐC THAM CHIẾU\n{'═'*88}")
print(f"  Tham chiếu, Qwen3-8B + self-eval : PA = {_ref['PA']:.2f}%   EA = {_ref['EA']:.2f}%")
if os.path.exists(_pub_csv):
    _pr = io_utils.score_saved_predictions(_pub_csv, test_all, "tham chiếu")
    _pm = pipeline.summarize(_pr, "tham chiếu")
    print(f"  Chính dự đoán đó, chấm lại      : PA_loose = {_pm['PA_loose']*100:.2f}%   "
          f"EA = {_pm['EA']*100:.2f}%")
    print(f"    (EA cao hơn vì executor cũ bỏ sót một phần câu table_*)")

for s in _main:
    if s.startswith("04_selfeval") or s.startswith("05_ace"):
        print(f"  {M[s]['label']:<32}: PA_loose = {M[s]['PA_loose']*100:.2f}%   "
              f"EA = {M[s]['EA']*100:.2f}%")

print(f"\n  ⚠ Chỉ PA_loose so trực tiếp được với cột PA tham chiếu.")
print(f"    EA tham chiếu tính bằng executor cũ nên thấp hơn thực tế.")

## §8. Xuất bảng và biểu đồ

In [ ]:
_csv = os.path.join(OUTPUT_DIR, f"bang_ket_qua_{STAMP}.csv")
with open(_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["nac", "mo_ta", "n", "EA", "PA_strict", "PA_loose",
                                      "no_program", "exec_none", "phut"])
    w.writeheader()
    for s in _main:
        m = M[s]
        w.writerow({"nac": s, "mo_ta": m["label"], "n": m["n"],
                    "EA": round(m["EA"]*100, 2), "PA_strict": round(m["PA_strict"]*100, 2),
                    "PA_loose": round(m["PA_loose"]*100, 2),
                    "no_program": round(m["no_program"]*100, 2),
                    "exec_none": round(m["exec_none"]*100, 2),
                    "phut": META.get(s, {}).get("metrics", {}).get("minutes", "")})

_cmp = os.path.join(OUTPUT_DIR, f"kiem_dinh_{STAMP}.csv")
with open(_cmp, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["so_sanh", "chi_so", "base", "variant", "delta",
                                      "ci_low", "ci_high", "p_value", "b", "c", "ket_luan"])
    w.writeheader()
    for c in comparisons:
        w.writerow({"so_sanh": c["label"], "chi_so": c["key"], "base": c["base"],
                    "variant": c["variant"], "delta": c["delta"],
                    "ci_low": c["ci95"][0], "ci_high": c["ci95"][1],
                    "p_value": round(c["p_value"], 5), "b": c["b"], "c": c["c"],
                    "ket_luan": c["verdict"]})
print(f"[SAVE] {_csv}\n[SAVE] {_cmp}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

_short = [M[s]["label"].split("—")[0].strip() for s in _main]
x = np.arange(len(_main))
for i, (k, lbl) in enumerate([("EA", "EA"), ("PA_strict", "PA")]):
    vals = [M[s][k] for s in _main]
    bars = axes[0].bar(x + i*0.38, vals, 0.38, label=lbl)
    for b, v in zip(bars, vals):
        axes[0].text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=7)
axes[0].set_xticks(x + 0.19); axes[0].set_xticklabels(_short, rotation=20, ha="right", fontsize=8)
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)
axes[0].set_title(f"Thang bậc — ViNumQA test ({len(test_all)} mẫu)")

_ea = [c for c in comparisons if c["key"] == "ea" and not c["base"].endswith("random_base")]
if _ea:
    lab = [f"{c['variant'][:14]}\nvs {c['base'][:14]}" for c in _ea]
    dl = [c["delta"] for c in _ea]
    er = [[c["delta"]-c["ci95"][0] for c in _ea], [c["ci95"][1]-c["delta"] for c in _ea]]
    col = ["tab:green" if c["p_value"] < 0.05 and c["delta"] > 0 else
           "tab:red" if c["p_value"] < 0.05 else "tab:gray" for c in _ea]
    axes[1].bar(lab, dl, color=col, yerr=er, capsize=4)
    for i, c in enumerate(_ea):
        axes[1].text(i, c["delta"], f"p={c['p_value']:.3f}", ha="center",
                     va="bottom" if c["delta"] >= 0 else "top", fontsize=7)
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_ylabel("ΔEA (thanh = KTC 95%)")
axes[1].tick_params(axis="x", labelsize=7); axes[1].grid(axis="y", alpha=0.3)
axes[1].set_title("Phần tăng thêm của từng kỹ thuật")

for s in _main:
    bs = M[s]["by_steps"]
    ks = sorted(bs, key=int)
    axes[2].plot(ks, [bs[k][1]/bs[k][0] for k in ks], marker="o",
                 label=M[s]["label"].split("—")[0].strip())
axes[2].set_xlabel("số phép toán trong gold"); axes[2].set_ylabel("EA")
axes[2].set_ylim(0, 1); axes[2].legend(fontsize=7); axes[2].grid(alpha=0.3)
axes[2].set_title("EA theo độ phức tạp")

plt.tight_layout()
_png = os.path.join(OUTPUT_DIR, f"bao_cao_{STAMP}.png")
plt.savefig(_png, dpi=150); plt.show()
print(f"[SAVE] {_png}")

## §9. Đọc kết quả cho đúng

| Điều cần nhớ | Vì sao |
|---|---|
| Chỉ **`PA_loose`** so trực tiếp được với bảng tham chiếu | `PA_strict` chặt hơn; EA cũ tính bằng executor lỗi `table_*` |
| Chênh lệch < ~2 điểm EA thường **không** đạt p < 0.05 | n = 497 là nhỏ. Muốn chắc hơn: chạy thêm trên valid (584 mẫu) rồi gộp |
| Không trộn kết quả giữa máy có cắt ngữ cảnh và máy không cắt | mỗi notebook in rõ ở dòng `[PROMPT]` |
| Chạy nhiều cấu hình thì dễ có cái "đạt p<0.05" do may mắn | kết luận mạnh chỉ nên dựa vào các bước leo thang đã định trước |
| Nấc 4 và nấc 5 đều là cơ chế sửa lỗi lúc suy luận | nếu ACE ≈ self-eval thì nhiều khả năng **chồng lấn**, không phải ACE vô dụng |

## Cái giá của từng kỹ thuật

Ngoài EA/PA, dự án đặt trong bối cảnh **tài nguyên hạn chế**, nên nên báo cáo kèm:

| Nấc | Lượt sinh / mẫu | Cần huấn luyện? | Cần API ngoài? |
|---|:---:|:---:|:---:|
| 1, 2 | 1 | ❌ | ❌ |
| 3 (SFT) | 1 | ✅ một lần | ❌ |
| 4 (self-eval) | 2 | ❌ | ❌ |
| 5 (ACE) | 2 + pha A một lần | ❌ | ❌ |

Cột cuối là điểm mạnh chung của cả lộ trình: **không nấc nào cần API ngoài**, nên toàn bộ
nằm trong thiết lập *constrained-resource* của dự án — khác với nhánh Phi-4 + Gemini
(unconstrained) vốn còn cho kết quả thấp hơn.